# KD-ReID — Colab 執行器

**程式碼以 GitHub 為準，Colab 只負責「拉最新程式 → 訓練」。** 不要在 Colab 裡改 `kdreid/` 的程式碼（runtime 重置就會消失）；要改就在 Claude / 本機改完 push，這裡重新執行第 3 步就會 `git pull`。

每次開新的 runtime 時從上到下執行一次。

**事前準備（只需做一次）**
1. Runtime → Change runtime type → **GPU**（T4 即可，A100/L4 更快）
2. 資料集 zip 放在 Drive `MyDrive/KD-ReID/datasets/`（檔名不限，例如 `archive.zip`）
3. 這是私有 repo：在 Colab 左側 🔑 **Secrets** 新增 `GH_TOKEN`（GitHub fine-grained token，只給 `cts0424/KD-ReID` 的 Contents read 權限），並開啟 Notebook access

## 1. 檢查 GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| CUDA", torch.cuda.is_available())

## 2. 掛載 Google Drive（存放資料集壓縮檔與 checkpoints）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/KD-ReID/outputs /content/drive/MyDrive/KD-ReID/datasets

## 3. Clone repo 並安裝

Colab 已內建 CUDA 版的 torch，`pip install -e .` 不會重裝它。

In [ ]:
import os
from google.colab import userdata
token = userdata.get('GH_TOKEN')
REPO = '/content/KD-ReID'
if not os.path.exists(REPO):
    !git clone https://{token}@github.com/cts0424/KD-ReID.git {REPO}
else:
    !git -C {REPO} pull --ff-only
%cd {REPO}
!pip install -q -e ".[dev,log]"

## 4. 把資料集複製到本機磁碟並解壓

直接從 Drive 讀上萬張小圖會非常慢，所以訓練時一律從 `/content/data` 讀。程式會自動在 `/content/data` 底下找到 `bounding_box_train` 所在的資料夾。

In [ ]:
import glob, os, subprocess
DS_DIR = '/content/drive/MyDrive/KD-ReID/datasets'
zips = sorted(glob.glob(f'{DS_DIR}/*.zip'))
assert zips, f'No .zip found in {DS_DIR}'
os.makedirs('/content/data', exist_ok=True)
if not glob.glob('/content/data/**/bounding_box_train', recursive=True):
    for z in zips:
        print('unzipping', z)
        subprocess.run(['cp', z, '/content/'], check=True)
        subprocess.run(['unzip', '-q', '-o', f'/content/{os.path.basename(z)}', '-d', '/content/data'], check=True)
print(*glob.glob('/content/data/**/bounding_box_train', recursive=True), sep='\n')

## 5. Smoke test（約 10 秒，確認環境正常）

In [ ]:
!python -m pytest -q

## 6. 訓練

依序執行：
1. **Teacher**（ResNet-50）→ `outputs/teacher_r50/best.pth`
2. **Student baseline**（ResNet-18，無蒸餾）→ 比較基準
3. **Student + KD**（ResNet-18 ← ResNet-50）

輸出寫在 Drive 上，斷線後重新執行同一行會從 `last.pth` 自動接續。

In [ ]:
!python tools/train.py --config configs/teacher_r50.yaml

In [ ]:
!python tools/train.py --config configs/student_r18_baseline.yaml

In [ ]:
!python tools/train.py --config configs/student_r18_kd.yaml

## 7. 查看訓練日誌

每個 output_dir 都有 `log.txt`、`config.yaml`、`last.pth`、`best.pth`。

In [ ]:
!tail -n 20 /content/drive/MyDrive/KD-ReID/outputs/student_r18_kd/log.txt